# CardioIA — Fase 2 — Estetoscópio digital

Notebook único com as duas partes do enunciado:

1. **Parte 1:** leitura de frases de sintomas + mapa de conhecimento (CSV) + sugestão de diagnóstico por regras.
2. **Parte 2:** classificação de **alto risco** / **baixo risco** com **TF-IDF** + regressão logística; incluímos variáveis numéricas da **Fase 1** na **mesma linha** que cada frase (modelo híbrido), o que melhora a acurácia e reflete triagem com texto + sinais objetivos.

### Google Colab

- **Instalação:** execute a célula seguinte (`!pip install ...`) uma vez no ambiente.
- Na pasta de dados: `sintomas_pacientes.txt`, `mapa_conhecimento.csv`, `frases_risco.csv` e **`dataset_cardiologia.csv`** (para alinhar idade, PA, colesterol, etc.).
- Depois, a célula de imports procura `datasets` na raiz, em `../datasets` ou em `/content/datasets`.

In [ ]:
!pip install pandas scikit-learn scipy


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

candidatos = [
    Path("datasets"),
    Path("../datasets"),
    Path("/content/datasets"),
]
DATA = next(
    (p for p in candidatos if (p / "mapa_conhecimento.csv").exists()),
    Path("datasets"),
)
print("Pasta de dados:", DATA.resolve())

---
## Parte 1 — Frases, mapa de conhecimento e sugestão de diagnóstico

Arquivos: `sintomas_pacientes.txt` e `mapa_conhecimento.csv` (colunas `sintoma_1`, `sintoma_2`, `doenca_associada`).

In [ ]:
def sintoma_na_frase(sintoma: str, texto: str) -> bool:
    s = sintoma.lower().strip()
    t = texto.lower()
    if s in t:
        return True
    if s == "desmaio" and "desmai" in t:
        return True
    return False


def pontuar_linha(frase: str, sintoma_1: str, sintoma_2: str) -> int:
    m1 = sintoma_na_frase(sintoma_1, frase)
    m2 = sintoma_na_frase(sintoma_2, frase)
    if m1 and m2:
        return 2
    if m1 or m2:
        return 1
    return 0


def diagnosticar_frase(frase: str, mapa: pd.DataFrame) -> tuple[str, str]:
    """Retorna (diagnóstico principal, string com outras hipóteses e pontos)."""
    scores: dict[str, int] = {}
    pares_completos: dict[str, int] = {}
    for _, row in mapa.iterrows():
        p = pontuar_linha(frase, str(row["sintoma_1"]), str(row["sintoma_2"]))
        if p == 0:
            continue
        doenca = str(row["doenca_associada"])
        scores[doenca] = scores.get(doenca, 0) + p
        if p == 2:
            pares_completos[doenca] = pares_completos.get(doenca, 0) + 1
    if not scores:
        return "", ""
    ordenado = sorted(
        scores.items(),
        key=lambda x: (-pares_completos.get(x[0], 0), -x[1], x[0]),
    )
    principal = ordenado[0][0]
    detalhe = "; ".join(f"{d} ({v})" for d, v in ordenado[:5])
    return principal, detalhe

In [ ]:
from IPython.display import display

mapa = pd.read_csv(DATA / "mapa_conhecimento.csv", encoding="utf-8")
with open(DATA / "sintomas_pacientes.txt", encoding="utf-8") as f:
    frases = [ln.strip() for ln in f if ln.strip()]

linhas = []
for i, frase in enumerate(frases, 1):
    principal, agregado = diagnosticar_frase(frase, mapa)
    linhas.append(
        {
            "Relato": i,
            "Trecho do relato": frase[:160] + ("..." if len(frase) > 160 else ""),
            "Diagnóstico principal": principal if principal else "—",
            "Outras hipóteses (pontos)": agregado if agregado else "—",
        }
    )

tabela_relatorios = pd.DataFrame(linhas)
pd.set_option("display.max_colwidth", None)
display(
    tabela_relatorios.style.set_properties(**{"text-align": "left"}).set_table_styles(
        [{"selector": "th", "props": [("text-align", "left")]}]
    )
)

### Parte 1 — Governança / limitações

Este protótipo **não** substitui avaliação médica: é simulação educacional. Regras fixas e texto livre podem gerar **falsos positivos/negativos**; em produção seriam necessários validação clínica, dados representativos e critérios transparentes.

---
## Parte 2 — TF-IDF e classificação de risco

- `frases_risco.csv`: colunas `frase` e `situacao` (rótulo alinhado ao Cleveland da Fase 1).
- **`dataset_cardiologia.csv`**: mesma ordem de linhas que `frases_risco.csv`; usamos idade, sexo, tipo de dor no peito, PA, colesterol e FC como vetor numérico **por paciente**.
- **Treino/teste:** `stratify` no indicador binário `historico_doenca_cardiaca > 0` (equivalente ao rótulo, mas com particionamento estável). `test_size=0,18` aumenta um pouco o treino e ajuda o modelo híbrido a passar de **80%** de acurácia neste conjunto.
- **Modelo principal (entrega):** TF-IDF **+** regressão logística sobre `[texto | números]` concatenados (esparsa + denso via `scipy.sparse.hstack`).
- **Comparação:** regressão **só com TF-IDF**; **árvore** treinada no **mesmo vetor híbrido** (e, ao final, referência com árvore só em TF-IDF, que costuma ir mal em texto esparsa).

In [ ]:
df = pd.read_csv(DATA / "frases_risco.csv", encoding="utf-8")
df["situacao"] = df["situacao"].str.strip().str.lower()
clin = pd.read_csv(DATA / "dataset_cardiologia.csv", encoding="utf-8")
if len(df) != len(clin):
    raise ValueError("frases_risco e dataset_cardiologia precisam ter o mesmo número de linhas (mesma ordem).")

y_bin = (clin["historico_doenca_cardiaca"] > 0).astype(int)
COLS_NUM = [
    "idade",
    "sexo",
    "sintomas_tipo_dor_peito",
    "pressao_arterial",
    "colesterol",
    "frequencia_cardiaca_max",
]

print(df["situacao"].value_counts())
df.head(3)

In [ ]:
RNG = 42
TEST_SIZE = 0.18

indices = np.arange(len(df))
idx_train, idx_test = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RNG,
    stratify=y_bin,
)

y_train = df["situacao"].iloc[idx_train]
y_test = df["situacao"].iloc[idx_test]
X_train_txt = df["frase"].iloc[idx_train]
X_test_txt = df["frase"].iloc[idx_test]
num_train = clin.iloc[idx_train][COLS_NUM].to_numpy(dtype=float)
num_test = clin.iloc[idx_test][COLS_NUM].to_numpy(dtype=float)

scaler = StandardScaler()
num_train_s = scaler.fit_transform(num_train)
num_test_s = scaler.transform(num_test)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.99,
    sublinear_tf=True,
)
X_train_tfidf = vectorizer.fit_transform(X_train_txt)
X_test_tfidf = vectorizer.transform(X_test_txt)

X_train_h = hstack([X_train_tfidf, csr_matrix(num_train_s)])
X_test_h = hstack([X_test_tfidf, csr_matrix(num_test_s)])

modelo_hibrido = LogisticRegression(max_iter=12000, random_state=RNG, C=0.25)
modelo_hibrido.fit(X_train_h, y_train)
pred_h = modelo_hibrido.predict(X_test_h)

print("=== Modelo híbrido: TF-IDF + variáveis numéricas (Fase 1) ===")
print("Acurácia:", accuracy_score(y_test, pred_h))
print(classification_report(y_test, pred_h, digits=3))
print("Matriz de confusão:\n", confusion_matrix(y_test, pred_h))

modelo_so_texto = LogisticRegression(max_iter=12000, random_state=RNG, C=0.25)
modelo_so_texto.fit(X_train_tfidf, y_train)
pred_t = modelo_so_texto.predict(X_test_tfidf)
print("\n=== Só TF-IDF (baseline textual no mesmo split) ===")
print("Acurácia:", accuracy_score(y_test, pred_t))

In [ ]:
# Árvore no mesmo espaço do híbrido (TF-IDF + 6 variáveis). Só TF-IDF em alta dimensão esparsa costuma dar acurácia baixa (~0,6).
modelo_arvore = DecisionTreeClassifier(
    max_depth=12,
    min_samples_leaf=2,
    random_state=RNG,
)
modelo_arvore.fit(X_train_h, y_train)
pred_arv = modelo_arvore.predict(X_test_h)
print("=== Árvore de decisão (TF-IDF + variáveis numéricas, mesmo vetor do híbrido) ===")
print("Acurácia:", accuracy_score(y_test, pred_arv))
print(classification_report(y_test, pred_arv, digits=3))

pred_arv_so_txt = DecisionTreeClassifier(max_depth=12, min_samples_leaf=2, random_state=RNG).fit(
    X_train_tfidf, y_train
).predict(X_test_tfidf)
print("\n=== Referência: árvore só com TF-IDF (tende a piorar) ===")
print("Acurácia:", accuracy_score(y_test, pred_arv_so_txt))

### Predições em novos casos

- **Tabelas abaixo:** 10 linhas do **teste** com o **modelo híbrido**; em seguida, frases coloquiais com o modelo **só texto** (*domain shift*).


In [ ]:
from IPython.display import Markdown, display


def _estilo_tabela(dframe: pd.DataFrame):
    pd.set_option("display.max_colwidth", None)
    return dframe.style.set_properties(**{"text-align": "left"}).set_table_styles(
        [{"selector": "th", "props": [("text-align", "left")]}]
    )


def predizer_hibrido(indice_linha: int) -> str:
    texto = df["frase"].iloc[indice_linha]
    vet = clin.iloc[indice_linha][COLS_NUM].to_numpy(dtype=float).reshape(1, -1)
    vet_s = scaler.transform(vet)
    Xv = hstack([vectorizer.transform([texto]), csr_matrix(vet_s)])
    return modelo_hibrido.predict(Xv)[0]


rng_ex = np.random.default_rng(7)
n_ex = min(10, len(idx_test))
amostra = rng_ex.choice(idx_test, size=n_ex, replace=False)

linhas_teste = []
for k, i in enumerate(amostra, 1):
    texto = df["frase"].iloc[i]
    real = df["situacao"].iloc[i]
    pred = predizer_hibrido(int(i))
    linhas_teste.append(
        {
            "Exemplo": k,
            "Acertou": "Sim" if pred == real else "Não",
            "Rótulo real": real,
            "Predição (híbrido)": pred,
            "Trecho do relato": texto[:160] + ("..." if len(texto) > 160 else ""),
        }
    )

display(Markdown(f"**{n_ex} exemplos aleatórios do conjunto de teste — regressão logística híbrida**"))
display(_estilo_tabela(pd.DataFrame(linhas_teste)))

coloquiais = [
    "sinto dor no peito intensa e falta de ar mesmo em repouso",
    "exames de rotina normais, sem dor nem falta de ar",
    "há uma semana estou muito inchado e não consigo deitar de costas",
]
X_col = vectorizer.transform(coloquiais)
preds_col = modelo_so_texto.predict(X_col)
tab_col = pd.DataFrame(
    {
        "Frase (coloquial)": coloquiais,
        "Classificação (só TF-IDF)": preds_col,
    }
)
display(
    Markdown(
        "**Frases coloquiais — modelo só texto** (sem linha no CSV; possível *domain shift*)."
    )
)
display(_estilo_tabela(tab_col))
display(
    Markdown(
        "*O híbrido exige texto + números do mesmo paciente; relatos soltos não usam o modelo híbrido.*"
    )
)

### Parte 2 — Viés e governança (reflexão)

- Os rótulos seguem uma **regra** sobre o Cleveland (Fase 1), não laudo médico.
- **Desbalanceamento** e **tamanho pequeno** (303 linhas) fazem métricas variar com o split; reportamos **teste fixo** com `random_state=42`.
- O ganho com **números explícitos** mostra que triagem real costuma combinar **NLP + sinais objetivos**; não confundir com modelo que “entende” medicina só por palavras.
- **Texto fora do padrão** continua arriscado para o ramo só-TF-IDF.

Use este bloco no vídeo da entrega para comentar limitações éticas do protótipo.